# AURA: Download your saved export

**Run only the next cell.** It mounts Drive, reads the existing export's manifest,
verifies every file, and creates one ZIP with a browser download. It needs no GPU,
model loading, training, previous notebook variables or additional Python uploads.

The source is the export already saved in Drive, not the incomplete local download.
Keep `EXPORT_FOLDER_NAME` set to the run you want. The original files stay untouched.
If Drive itself is missing a file or a hash differs, the cell lists the problem and
refuses to publish an incomplete ZIP. That error does not mean training must be rerun.

Wait for **“ZIP ready”** and for the browser download to finish. Extract the resulting
`-verified` folder into **`AURA/artifacts/demo_packages/`**. Keep this new folder separate
from the earlier partial download. If the browser download is blocked, the same single
verified ZIP is saved under `MyDrive/AURA_Colab_Demo/downloads/` for direct download.

In [ ]:
"""Create a verified download from an existing Drive export, without model loading."""

import hashlib
import json
import zipfile
from pathlib import Path


def create_verified_download(source: str | Path, destination: str | Path) -> Path:
    """Verify every declared file while writing one ZIP; reject incomplete exports."""
    source = Path(source).resolve()
    destination = Path(destination).resolve()
    manifest_path = source / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"No saved export manifest in {source}.")
    manifest_bytes = manifest_path.read_bytes()
    manifest = json.loads(manifest_bytes)
    if manifest.get("schema_version") != 1 or not manifest.get("file_sha256"):
        raise ValueError("The saved export has an unsupported or empty manifest.")
    expected = manifest["file_sha256"]
    missing = []
    for relative in expected:
        path = source / relative
        if source not in path.resolve().parents:
            raise ValueError(f"Invalid export file path: {relative}")
        if not path.is_file():
            missing.append(relative)
    if missing:
        raise FileNotFoundError(
            f"The Drive export itself is incomplete: {len(missing)} of {len(expected)} "
            "files are missing. No ZIP was created. Missing files:\n" + "\n".join(missing)
        )
    partial = destination.with_name(destination.name + ".partial")
    if destination.exists() or partial.exists():
        raise FileExistsError("Choose a new ZIP filename; previous downloads are preserved.")
    destination.parent.mkdir(parents=True, exist_ok=True)
    folder_name = source.name + "-verified"
    print(f"Found all {len(expected)} files. Verifying and archiving them…", flush=True)
    try:
        # Use a low compression level to reduce download size without a long
        # compression pass. A temporary suffix prevents downloading a partial ZIP.
        with zipfile.ZipFile(
            partial, "x", compression=zipfile.ZIP_DEFLATED, compresslevel=1, allowZip64=True
        ) as archive:
            for index, (relative, wanted) in enumerate(sorted(expected.items()), start=1):
                path = source / relative
                if path.stat().st_size > 50_000_000 or index % 15 == 0:
                    print(f"[{index}/{len(expected)}] {relative}", flush=True)
                digest = hashlib.sha256()
                with (
                    path.open("rb") as incoming,
                    archive.open(f"{folder_name}/{relative}", "w", force_zip64=True) as outgoing,
                ):
                    for block in iter(lambda: incoming.read(1024 * 1024), b""):
                        digest.update(block)
                        outgoing.write(block)
                if digest.hexdigest() != wanted:
                    raise ValueError(
                        f"Saved Drive file differs from its manifest: {relative}. "
                        "No downloadable ZIP was published."
                    )
            if manifest_path.read_bytes() != manifest_bytes:
                raise ValueError(
                    "The export manifest changed while copying. Retry once it is stable."
                )
            archive.writestr(f"{folder_name}/manifest.json", manifest_bytes)
        partial.rename(destination)
    except BaseException:
        partial.unlink(missing_ok=True)
        raise
    print(f"Verified {len(expected)}/{len(expected)} files. ZIP ready: {destination}", flush=True)
    return destination


from datetime import datetime, timezone
from google.colab import drive, files

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/AURA_Colab_Demo")
EXPORT_FOLDER_NAME = "aura-demo-20261102T173046496634Z-181716275116"
SAVED_EXPORT = DRIVE_ROOT / "exports" / EXPORT_FOLDER_NAME
if not SAVED_EXPORT.is_dir():
    available = sorted(p.parent.name for p in (DRIVE_ROOT / "exports").glob("*/manifest.json"))
    raise FileNotFoundError(
        f"Export folder not found: {SAVED_EXPORT}. "
        f"Available exports: {available}. Update EXPORT_FOLDER_NAME to one of these."
    )
DOWNLOAD_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
VERIFIED_ZIP = DRIVE_ROOT / "downloads" / f"{EXPORT_FOLDER_NAME}-{DOWNLOAD_STAMP}-verified.zip"
VERIFIED_ZIP = create_verified_download(SAVED_EXPORT, VERIFIED_ZIP)
print(f"Download size: {VERIFIED_ZIP.stat().st_size / (1024 ** 3):.2f} GiB")
print("Starting the browser download. Keep this Colab tab open until it finishes.")
files.download(str(VERIFIED_ZIP))